In [ ]:
import argparse
import math
import pysam
import numpy as np
import pandas as pd
import sys

from Bio import SeqIO
from tqdm import tqdm
from datetime import datetime
from multiprocessing import Pool
from concurrent.futures import ThreadPoolExecutor, ProcessPoolExecutor
from itertools import repeat

In [ ]:
working_path = "/home/wbguo/iproject/BSReadSim/test/"

ref_fasta = working_path + "/data/ref/GRCh38_full_analysis_set_plus_decoy_hla.fa"
wgs_file  = working_path + "/data/WGS/ERR1726424.mkdup.sorted.bam"
wgbs_file = working_path + "/data/WGBS/ERR2359938.mkdup.sorted.bam"

In [ ]:
ref_dict  = SeqIO.to_dict(SeqIO.parse(ref_fasta, "fasta"))

In [ ]:
contig_id  = 'chrM'
contig_seq = ref_dict[contig_id].seq.upper()
contig_len = len(contig_seq)
window_size= 100
num_windows= math.ceil(contig_len / window_size)

In [ ]:
pos_arr = np.array(range(0, contig_len, window_size))
window_list= [(pos_arr[ix], pos_arr[ix+1]) for ix in range(len(pos_arr)-1)] + [(pos_arr[-1], contig_len)]
sys.stderr.write(f'UTC Time: {datetime.utcnow()}\n')

In [ ]:
window_list

# ratio

In [ ]:
ratio_arr = np.zeros(num_windows)
for idx, pos in enumerate(range(0, contig_len, window_size)):
    pos_l = pos
    pos_r = pos + window_size

    tmp_seq = contig_seq[pos_l:pos_r] # check a the boundary
    ratio_arr[idx] = (tmp_seq.count("C") + tmp_seq.count("G"))/window_size
sys.stderr.write(f'UTC Time: {datetime.utcnow()}\n')

In [ ]:
def calculate_avg_depth(window, bam_file):
    start, end = window
    bam_obj  = pysam.AlignmentFile(bam_file, "rb") # must be in the loop, otherwise will have problem
    # important to set truncate and stepper
    pileupcol= bam_obj.pileup(contig_id, start, end, truncate = True, stepper = 'all', min_mapping_quality = 30)
    avg_depth= np.array([col.n for col in pileupcol]).mean()
    return avg_depth

def calculate_avg_depth_helper(args):
    return calculate_avg_depth(*args)

def calculate_avg_depth_parallel(window_list, bam_file):
    # contig level parallalization
    with ProcessPoolExecutor(max_workers=12) as pool:
        depth_arr = list(pool.map(calculate_avg_depth_helper, zip(window_list, repeat(bam_file))))
    return depth_arr

# frag length distribution

# WGS

In [ ]:
wgs_arr = calculate_avg_depth_parallel(window_list, wgs_file)
sys.stderr.write(f'UTC Time: {datetime.utcnow()}\n')

In [ ]:
wgbs_arr= calculate_avg_depth_parallel(window_list, wgbs_file)
sys.stderr.write(f'UTC Time: {datetime.utcnow()}\n')

# set truncate

In [ ]:
wgs_bam = pysam.AlignmentFile(wgs_file, "rb")

In [ ]:
wgbs_bam = pysam.AlignmentFile(wgbs_file, "rb")

In [ ]:
for pileupcol in wgs_bam.pileup(contig_id, start=10**7, end=10**7+100, stepper = 'all', min_mapping_quality = 30):
    print("Depth at position", pileupcol.pos, "is", pileupcol.n)

In [ ]:
for pileupcol in wgs_bam.pileup(contig_id, start=10**7, end=10**7+100, truncate = True, stepper = 'all', min_mapping_quality = 30):
    print("Depth at position", pileupcol.pos, "is", pileupcol.n)